# 8장 Semantic Search and Retrieval-Augmented Generation — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter08/Chapter 8 - Semantic Search.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter08/Chapter%208%20-%20Semantic%20Search.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter08` 노트북(Apache 2.0)을 Studio에서 돌아가도록 수정한 것입니다. 단어가 일치하는 문서가 아니라 **뜻이 가까운** 문서를 찾는 시맨틱 검색(semantic search)과, 찾은 문서를 LLM에게 근거로 주고 답하게 하는 **RAG**(Retrieval-Augmented Generation, 검색 증강 생성)를 다룹니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | **밀집 검색(dense retrieval)**: 문장 임베딩 + FAISS 최근접 검색, BM25 키워드 검색과 비교 (+ 거리 임계값, 청크 나누기) | 319–336 |
| 2 | **리랭킹(reranking)**: 1차 검색 결과를 관련도 점수로 다시 정렬 | 336–341 |
| 3 | 검색 평가 지표 MAP (+ 세 검색 방식 점수 비교) | 341–349 |
| 4 | **RAG**: API로 근거 기반 생성, 로컬 Phi-3 + FAISS로 RAG 파이프라인 (+ 주행 문서 RAG, 근거 없을 때의 동작) | 349–358 |
| 5 | 고급 RAG 기법과 RAG 평가 정리 | 355–358 |

**권장 환경:** `ml.g4dn.xlarge`(T4 16GB), 스토리지 30GB 이상(4비트 GGUF 2.4GB + 작은 임베딩·리랭커 모델). 1–3절은 GPU 없이도 돌아갑니다.

**API 키 정책.** 책은 1–3절 임베딩·리랭킹과 4절 생성에 유료 Cohere API를 씁니다. 이 노트북은 키가 없으면 **로컬 무료 모델로 자동 대체**합니다. 임베딩은 `BAAI/bge-small-en-v1.5`(책 4절에서 쓰는 모델), 리랭커는 `cross-encoder/ms-marco-MiniLM-L-6-v2`(책 밖)입니다. `COHERE_API_KEY` 환경변수가 있으면 책과 같은 Cohere 경로를 탑니다. Amazon Bedrock 대체 셀(책 밖)도 넣었습니다.

`(+ …)` 표시는 책에 없는 보충 실험이고, **(책 밖)** 표시는 책 출간(2024) 이후 내용입니다.

## 0. 환경 점검과 패키지 설치

LangChain 계열은 7장과 같은 버전 조합(현재 버전 + `langchain-classic`)을 씁니다. 이유는 7장 노트북 첫머리에 적었습니다. 설치 후에는 커널을 재시작하세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import importlib.metadata as md_, subprocess, sys, os
pins = {
    "langchain-core": "1.7.6", "langchain-classic": "1.0.8", "langchain-community": "0.4.2",
    "langchain-huggingface": "1.2.3", "faiss-cpu": "1.15.1", "rank-bm25": "0.2.2",
    "sentence-transformers": None, "huggingface_hub": None,
}
if os.environ.get("COHERE_API_KEY"):
    pins["cohere"] = None
todo = []
for pkg, ver in pins.items():
    try:
        have = md_.version(pkg)
        if ver and have != ver: todo.append(f"{pkg}=={ver}")
    except md_.PackageNotFoundError:
        todo.append(f"{pkg}=={ver}" if ver else pkg)
if todo:
    print("설치:", todo)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *todo])
    print("설치 완료 → 커널을 재시작(Kernel > Restart)한 뒤 이 셀부터 다시 실행하세요.")
else:
    print("추가 설치 불필요")

## 1. 밀집 검색(Dense Retrieval) (PDF 319–336)

검색 대상 문서와 질문을 **같은 임베딩 공간**에 넣고, 질문 벡터와 가장 가까운 문서 벡터를 찾습니다(최근접 이웃, nearest neighbor). 2장에서 본 문장 임베딩을 검색에 쓰는 것입니다. "밀집(dense)"은 벡터 대부분의 값이 0이 아니라는 뜻으로, 단어 등장 여부로 만든 희소(sparse) 벡터를 쓰는 키워드 검색과 구분하는 말입니다.

### 1-1. 문서를 가져와 청크로 나누기

책과 같이 영화 *Interstellar* 위키백과 소개문을 마침표 기준으로 문장 단위로 나눕니다. 한 문장이 한 청크(chunk), 즉 검색 단위입니다.

In [ ]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

texts = text.split('.')
texts = [t.strip(' \n') for t in texts]
print(len(texts), "chunks")

> **책 코드 주의.** `split('.')`은 단순한 방법이라 "35 mm" 같은 문장은 괜찮지만 "Dr. Smith"처럼 약어가 있으면 잘못 자릅니다. 1-5절에서 다른 청크 방식을 비교합니다.

### 1-2. 청크 임베딩

`embed()` 함수 하나로 백엔드를 감쌌습니다. `COHERE_API_KEY`가 있으면 책처럼 Cohere API를, 없으면 로컬 `bge-small-en-v1.5`(384차원)를 씁니다.

`input_type`은 질문과 문서를 구분해 임베딩하라는 지시입니다. 짧은 질문과 긴 문서는 글의 성격이 달라서, 검색용 임베딩 모델은 둘을 다르게 처리하도록 학습됩니다. bge 모델은 질문 앞에 지시문을 붙이는 방식으로 같은 효과를 냅니다.

In [ ]:
import os, numpy as np
COHERE_EMBED_MODEL = "embed-english-v3.0"   # Cohere 문서에서 현재 모델명 확인
LOCAL_EMBED_MODEL = "BAAI/bge-small-en-v1.5"
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

if os.environ.get("COHERE_API_KEY"):
    import cohere
    co = cohere.Client(os.environ["COHERE_API_KEY"])
    def embed(items, input_type):
        return np.array(co.embed(texts=items, input_type=input_type, model=COHERE_EMBED_MODEL).embeddings)
    EMBED_BACKEND = f"Cohere {COHERE_EMBED_MODEL}"
else:
    co = None
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer(LOCAL_EMBED_MODEL)
    def embed(items, input_type):
        if input_type == "search_query":
            items = [BGE_QUERY_PREFIX + q for q in items]
        return st_model.encode(items, normalize_embeddings=True)
    EMBED_BACKEND = f"local {LOCAL_EMBED_MODEL}"

embeds = embed(texts, input_type="search_document")
print(EMBED_BACKEND, embeds.shape)

### 1-3. 검색 인덱스 만들기

FAISS는 Meta가 만든 최근접 이웃 검색 라이브러리입니다. `IndexFlatL2`는 모든 벡터와 L2(유클리드) 거리를 하나하나 계산하는 가장 단순한 인덱스입니다. 문서가 수백만 개로 늘면 근사 인덱스(ANN, Approximate Nearest Neighbor)를 씁니다.

In [ ]:
import faiss
dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.float32(embeds))
index.ntotal

### 1-4. 인덱스 검색

In [ ]:
import pandas as pd

def search(query, number_of_results=3):
    query_embed = embed([query], input_type="search_query")[0]
    distances, similar_item_ids = index.search(np.float32([query_embed]), number_of_results)
    texts_np = np.array(texts)
    results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]], 'distance': distances[0]})
    print(f"Query:'{query}'\nNearest neighbors:")
    return results

pd.set_option("display.max_colwidth", 140)
query = "how precise was the science"
results = search(query)
results

**관찰 포인트.** 질문에 "science"만 겹치는데도 "scientific accuracy", "scientific consultant" 문장이 위로 올라옵니다. 뜻으로 찾는다는 것이 이것입니다.

> **책과 다를 수 있는 점.** 책의 거리 값(10757 등)은 Cohere의 4096차원 정규화되지 않은 벡터 기준입니다. 로컬 bge는 길이 1로 정규화한 벡터라 L2 거리가 0~2 범위입니다. 순위를 비교하세요. 절댓값은 모델마다 다릅니다.

이제 **키워드 검색(BM25)**과 비교합니다. BM25는 질문 단어가 문서에 얼마나 자주, 얼마나 드물게 나오는지로 점수를 매기는 전통적 방법입니다. 불용어(the, was 같은 단어)를 빼고 단어 단위로 맞춥니다.

In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string
from tqdm import tqdm

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

tokenized_corpus = [bm25_tokenizer(passage) for passage in tqdm(texts)]
bm25 = BM25Okapi(tokenized_corpus)

def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    print("Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

keyword_search(query="how precise was the science")

**관찰 포인트.** BM25는 "science"라는 글자가 그대로 들어간 문장(*science fiction film*, *The Science of Interstellar*)을 1·2위로 올립니다. 정작 "과학적 정확성" 문장은 단어가 "scientific"이라 못 찾습니다. 반대로 제품 번호나 고유명사처럼 **글자가 정확히 맞아야 하는** 질의에서는 키워드 검색이 강합니다. 그래서 실무에서는 둘을 합친 **하이브리드 검색**을 많이 씁니다.

### 1-5. 밀집 검색의 한계 (PDF 328–330)

문서에 답이 없는 질문을 던져도 밀집 검색은 무조건 "가장 가까운" 3개를 돌려줍니다.

In [ ]:
query = "What is the mass of the moon?"
results = search(query)
results

**(+ 보충) 거리 임계값.** 책이 제안하는 대처 중 하나는 거리가 일정 값보다 멀면 결과를 버리는 것입니다. 관련 있는 질문과 없는 질문의 1위 거리를 비교해 보세요. 임계값은 모델과 데이터마다 다르므로 실제로는 검증용 질문 세트로 정합니다.

In [ ]:
for q in ["how precise was the science", "who directed the film", "What is the mass of the moon?", "how to change a car tire"]:
    d, _ = index.search(np.float32(embed([q], "search_query")), 1)
    print(f"{d[0][0]:8.3f}  {q}")

### (+ 보충) 청크 나누기 비교 (PDF 330–334)

긴 문서는 임베딩 하나로 담기 어려워 잘라서 넣습니다. 책이 소개하는 방법은 문장 단위, 문단 단위, 고정 길이, 그리고 **앞뒤 청크와 겹치게(overlap)** 자르기입니다. 겹치게 자르면 경계에 걸린 문맥이 양쪽 청크에 모두 남습니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
paragraphs = [p.strip() for p in text.strip().split("\n\n")]
fixed = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0).split_text(text)
overlap = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=60).split_text(text)
pd.DataFrame({
    "method": ["sentence (book)", "paragraph", "fixed 200 chars", "fixed 200 + overlap 60"],
    "n_chunks": [len(texts), len(paragraphs), len(fixed), len(overlap)],
    "first_chunk": [texts[0][:70], paragraphs[0][:70], fixed[0][:70], overlap[1][:70]],
})

## 2. 리랭킹(Reranking) (PDF 336–341)

검색은 보통 여러 단계로 이뤄집니다. 1단계(BM25나 밀집 검색)는 빠르게 후보를 넉넉히 뽑고, 2단계 **리랭커**가 후보마다 질문과의 관련도 점수를 매겨 순서를 다시 정합니다.

리랭커는 대개 **크로스 인코더(cross-encoder)**입니다. 질문과 문서를 **한 입력으로 붙여** 모델에 넣고 0~1 관련도를 냅니다. 둘을 따로 임베딩하는 밀집 검색(바이 인코더, bi-encoder)보다 정확하지만, 문서마다 모델을 한 번씩 돌려야 해서 후보 수십 개에만 씁니다. 책은 이 방식을 monoBERT로 소개합니다.

`rerank()` 함수로 백엔드를 감쌌습니다. Cohere 키가 있으면 책의 `co.rerank`, 없으면 로컬 크로스 인코더(책 밖)를 씁니다. 점수 척도가 다릅니다. Cohere는 0~1 관련도를, 로컬 크로스 인코더는 시그모이드를 거치기 전의 로짓(logit, 음수일 수 있음)을 냅니다. 둘 다 **클수록 관련**이고, 이 질문처럼 막연한 질의에서는 Cohere 점수도 0.17 정도로 낮게 나옵니다(책 출력).

In [ ]:
COHERE_RERANK_MODEL = "rerank-v3.5"                           # Cohere 문서에서 현재 모델명 확인
LOCAL_RERANKER = "cross-encoder/ms-marco-MiniLM-L-6-v2"

if co is not None:
    def rerank(query, documents, top_n=3):
        r = co.rerank(query=query, documents=documents, top_n=top_n, return_documents=True, model=COHERE_RERANK_MODEL)
        return [(x.relevance_score, x.document.text) for x in r.results]
else:
    from sentence_transformers import CrossEncoder
    ce = CrossEncoder(LOCAL_RERANKER)
    def rerank(query, documents, top_n=3):
        scores = ce.predict([(query, d) for d in documents])     # 로짓: 클수록 관련 (Cohere는 0~1 점수)
        order = np.argsort(-scores)[:top_n]
        return [(float(scores[i]), documents[i]) for i in order]

query = "how precise was the science"
for idx, (score, doc) in enumerate(rerank(query, texts)):
    print(idx, f"{score:.3f}", doc)

BM25로 후보 10개를 뽑고 리랭커로 다시 정렬하는 책의 파이프라인입니다.

In [ ]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = sorted([{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n],
                       key=lambda x: x['score'], reverse=True)
    print("Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]
    print(f"\nTop-3 hits by rerank ({len(bm25_hits)} BM25 hits re-ranked)")
    for score, doc in rerank(query, docs, top_n=top_k):
        print("\t{:.3f}\t{}".format(score, doc.replace("\n", " ")))

keyword_and_reranking_search(query="how precise was the science")

**관찰 포인트.** BM25 1위였던 "science fiction film" 문장이 리랭킹 뒤에는 내려가고 Kip Thorne(과학 자문) 문장이 올라오는지 보세요. 리랭커는 1단계가 가져온 후보 안에서만 순서를 바꿉니다. 그래서 정답 문장("scientific accuracy")이 BM25 후보 10개에 아예 없으면 리랭커도 찾지 못합니다. 1단계의 재현율(recall)이 중요한 이유입니다.

## 3. 검색 평가: MAP (PDF 341–349)

검색 시스템을 평가하려면 **문서 모음, 질문 세트, 질문별 정답 문서 표시(relevance judgment)** 세 가지가 필요합니다.

- **precision@k:** 상위 k개 중 정답의 비율
- **AP(average precision):** 정답 문서가 나온 위치마다 precision@k를 구해 평균. 정답을 위쪽에 놓을수록 높습니다.
- **MAP(mean average precision):** 질문 세트 전체의 AP 평균

**(+ 보충)** 작은 질문 세트를 직접 만들어 밀집 검색, BM25, BM25+리랭킹을 비교합니다. 정답 표시는 사람이 판단해 정한 것입니다(문장 번호는 `texts` 순서).

In [ ]:
for i, t in enumerate(texts):
    print(i, t[:90])

In [ ]:
judgments = {   # 질문: 정답 문장 번호 집합 (사람이 판단)
    "how precise was the science": {4, 12},
    "how much money did the film make": {10},
    "where was the movie filmed": {6},
    "who wrote the script": {3},
    "what awards did it win": {14},
}

def average_precision(ranked_ids, relevant):
    hits, precs = 0, []
    for k, doc_id in enumerate(ranked_ids, start=1):
        if doc_id in relevant:
            hits += 1
            precs.append(hits / k)
    return sum(precs) / len(relevant) if relevant else 0.0

def rank_dense(q, k=5):
    return list(index.search(np.float32(embed([q], "search_query")), k)[1][0])
def rank_bm25(q, k=5):
    return list(np.argsort(-bm25.get_scores(bm25_tokenizer(q)))[:k])
def rank_bm25_rerank(q, k=5, cand=10):
    cands = list(np.argsort(-bm25.get_scores(bm25_tokenizer(q)))[:cand])
    reranked = rerank(q, [texts[i] for i in cands], top_n=k)
    return [texts.index(doc) for _, doc in reranked]
def rank_hybrid_rerank(q, k=5):
    cands = list(dict.fromkeys(rank_dense(q, 5) + rank_bm25(q, 5)))   # 두 방식 후보 합치기
    reranked = rerank(q, [texts[i] for i in cands], top_n=k)
    return [texts.index(doc) for _, doc in reranked]

systems = {"dense": rank_dense, "bm25": rank_bm25, "bm25+rerank": rank_bm25_rerank, "hybrid+rerank": rank_hybrid_rerank}
ap = pd.DataFrame({name: [average_precision(f(q), rel) for q, rel in judgments.items()] for name, f in systems.items()},
                  index=list(judgments))
ap.loc["MAP"] = ap.mean()
ap.round(3)

**읽는 법.** 행은 질문별 AP, 마지막 행이 MAP입니다. 리랭커를 붙인다고 항상 좋아지지는 않습니다. 작은 로컬 리랭커(MiniLM)와 한 문장짜리 청크에서는 밀집 검색만 쓴 쪽이 더 높게 나오기도 합니다(2026-11 CPU 실행에서 그랬습니다). 질문 5개는 통계적으로 의미 있는 크기도 아닙니다. 계산 방법을 익히는 용도로 보세요. 실제 평가는 수백 개 이상의 질문으로 합니다. 질문을 추가하거나 정답 표시를 바꿔 보면서 순위가 어떻게 흔들리는지도 확인해 보세요.

## 4. RAG: 검색 결과를 근거로 생성하기 (PDF 349–358)

LLM은 모르는 것도 그럴듯하게 지어냅니다(환각, hallucination). RAG는 **검색 → 찾은 문서를 프롬프트에 넣기 → 그 근거로 답하기** 순서로 이를 줄입니다. 답에 출처(citation)를 달 수 있다는 것도 장점입니다.

### 4-1. LLM API로 근거 기반 생성 (PDF 351–352)

책은 Cohere `co.chat(documents=...)`로 답과 함께 **문장 구간별 출처**를 받습니다. 키가 없으면 건너뜁니다.

In [ ]:
COHERE_CHAT_MODEL = "command-r-08-2024"   # Cohere 문서에서 현재 모델명 확인
query = "income generated"
results = search(query)

if co is not None:
    docs_dict = [{'text': t} for t in results['texts']]
    response = co.chat(message=query, documents=docs_dict, model=COHERE_CHAT_MODEL)
    print(response.text)
    print(response.citations)
else:
    print("COHERE_API_KEY가 없어 건너뜁니다. 아래 Bedrock 셀 또는 4-2절 로컬 RAG를 쓰세요.")

**(+ 보충, 책 밖) Amazon Bedrock으로 같은 일 하기.** Converse API에 검색된 문장을 번호와 함께 넣고, 답에 `[번호]`로 출처를 달라고 지시합니다. Studio 실행 역할에 Bedrock 호출 권한이 있고 콘솔에서 모델 접근이 켜져 있어야 합니다. 미국 밖 리전에서는 `global.` 접두사의 교차 리전 추론 ID를 씁니다.

In [ ]:
BEDROCK_MODEL_ID = "global.amazon.nova-2-lite-v1:0"   # 콘솔의 Bedrock > Model catalog에서 확인
BEDROCK_REGION = os.environ.get("AWS_REGION", "ap-northeast-2")

def bedrock_grounded_answer(question, passages):
    import boto3
    brt = boto3.client("bedrock-runtime", region_name=BEDROCK_REGION)
    ctx = "\n".join(f"[{i}] {p}" for i, p in enumerate(passages))
    prompt = (f"Answer using ONLY the passages below. Cite passage numbers like [0]. "
              f"If the passages do not contain the answer, say you don't know.\n\n{ctx}\n\nQuestion: {question}")
    r = brt.converse(modelId=BEDROCK_MODEL_ID, messages=[{"role": "user", "content": [{"text": prompt}]}],
                     inferenceConfig={"maxTokens": 300, "temperature": 0})
    return r["output"]["message"]["content"][0]["text"]

try:
    print(bedrock_grounded_answer(query, list(results["texts"])))
except Exception as e:
    print("Bedrock 사용 불가 →", type(e).__name__, str(e)[:200])

### 4-2. 로컬 모델로 RAG (PDF 352–355)

이번에는 외부 API 없이 **생성 모델(Phi-3 4비트 GGUF)**, **임베딩 모델(bge-small)**, **벡터 DB(FAISS)**를 모두 로컬에서 씁니다. 7장과 같은 llama.cpp 경로이고, llama-cpp-python 설치가 안 되면 `BACKEND="hf"`로 transformers를 씁니다(설치 방법은 7장 노트북 0절 참고).

> **책과 다를 수 있는 점.** 공식 노트북에 저장된 실행 로그는 다른 저장소의 8비트 파일(`Phi-3-mini-4k-instruct-Q8_0.gguf`, 3.8GB)을 받은 기록이지만, 코드는 Microsoft 저장소의 4비트 `q4` 파일을 받습니다. 여기서는 코드대로 `q4`를 씁니다.

In [ ]:
import sys, subprocess, importlib.util
from pathlib import Path
from huggingface_hub import hf_hub_download

BACKEND = "llamacpp"     # llama.cpp가 안 되면 "hf"
if BACKEND == "llamacpp" and importlib.util.find_spec("llama_cpp") is None and sys.version_info[:2] <= (3, 12):
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python==0.3.19",
                     "--only-binary=llama-cpp-python",
                     "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124"])

if BACKEND == "llamacpp":
    model_path = hf_hub_download("microsoft/Phi-3-mini-4k-instruct-gguf", "Phi-3-mini-4k-instruct-q4.gguf",
                                 local_dir=Path.home() / "models")
    from langchain_community.llms import LlamaCpp            # 책: from langchain import LlamaCpp
    llm = LlamaCpp(model_path=model_path, n_gpu_layers=-1, max_tokens=500, n_ctx=2048, seed=42, verbose=False)
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    from langchain_huggingface import HuggingFacePipeline
    hf_model = AutoModelForCausalLM.from_pretrained(
        "microsoft/Phi-3-mini-4k-instruct", device_map="cuda", torch_dtype="auto",
        trust_remote_code=False,   # 책 1장 본문은 True. 최신 transformers에는 Phi-3가 내장되어 False가 안전
    )
    hf_tok = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
    llm = HuggingFacePipeline(pipeline=pipeline("text-generation", model=hf_model, tokenizer=hf_tok,
                                                max_new_tokens=500, do_sample=False, return_full_text=False))
print(type(llm).__name__)

임베딩 모델과 벡터 DB를 준비합니다. LangChain의 `FAISS.from_texts`는 1절에서 직접 한 일(임베딩 → 인덱스 추가)을 한 줄로 해 줍니다.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings       # 책: from langchain.embeddings.huggingface import ...
from langchain_community.vectorstores import FAISS             # 책: from langchain.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(model_name='BAAI/bge-small-en-v1.5')
db = FAISS.from_texts(texts, embedding_model)
db.index.ntotal

RAG 프롬프트입니다. `{context}`에는 검색된 문서가, `{question}`에는 질문이 들어갑니다. `chain_type='stuff'`는 검색된 문서를 전부 프롬프트에 "채워 넣는" 가장 단순한 방식입니다.

In [ ]:
from langchain_core.prompts import PromptTemplate              # 책: from langchain import PromptTemplate
from langchain_classic.chains import RetrievalQA               # 책: from langchain.chains import RetrievalQA

template = """<|user|>
Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""
prompt = PromptTemplate(template=template, input_variables=["context", "question"])

rag = RetrievalQA.from_chain_type(
    llm=llm, chain_type='stuff', retriever=db.as_retriever(),
    chain_type_kwargs={"prompt": prompt}, verbose=True,
)
rag.invoke('Income generated')

**(+ 보충) 현재 권장 방식(LCEL)으로 같은 RAG.** `RetrievalQA`가 내부에서 하는 일을 펼쳐 쓰면 "검색기 → 문서 이어 붙이기 → 프롬프트 → LLM"입니다. 검색된 문서도 함께 돌려받도록 해서 근거를 확인할 수 있게 했습니다.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
retriever = db.as_retriever(search_kwargs={"k": 3})
join_docs = lambda docs: "\n".join(d.page_content for d in docs)

rag_lcel = RunnableParallel(docs=retriever, question=RunnablePassthrough()).assign(
    answer=(lambda x: {"context": join_docs(x["docs"]), "question": x["question"]}) | prompt | llm
)
out = rag_lcel.invoke("Who was the scientific consultant?")
print("ANSWER:", out["answer"].strip())
for d in out["docs"]:
    print(" - source:", d.page_content[:100])

### (+ 보충) 주행 문서로 RAG 해 보기

자율주행 개발 조직에서 흔한 질문, 즉 "우리 시험 규정에 따르면 …?"를 흉내 냅니다. 아래 문서는 **설명용으로 지어낸 가상의 사내 시험 규정**이고 실제 법규나 표준이 아닙니다. 두 가지를 확인하세요.

1. 답이 문서에 있는 질문: 검색된 근거로 답하는가
2. 답이 문서에 없는 질문: 프롬프트의 "근거가 없으면 모른다고 답하라" 지시를 따르는가. 작은 모델은 이 지시를 자주 어기고 그럴듯한 숫자를 지어냅니다. RAG를 써도 환각이 0이 되지는 않는다는 점을 직접 보는 실험입니다.

한국어 문서라면 `bge-small-en`(영어 전용) 대신 다국어 임베딩 모델(예: `BAAI/bge-m3`, `intfloat/multilingual-e5-small`)을 쓰세요.

In [ ]:
drive_docs = [
    "[TP-01] Closed-course tests must start with a sensor calibration check of all cameras and the lidar; a failed check aborts the run.",
    "[TP-02] During highway-pilot tests the safety driver must keep hands within 5 cm of the wheel and take over if the system requests it within 4 seconds.",
    "[TP-03] Night tests are allowed only when the ODD log confirms dry road surface and visibility above 200 m.",
    "[TP-04] Any disengagement must be logged with timestamp, GPS position, and a 30-second sensor snapshot before and after the event.",
    "[TP-05] School-zone scenarios are tested at a maximum of 30 km/h with a pedestrian dummy crossing from behind a parked vehicle.",
    "[TP-06] Rain tests require the wiper-state signal to be recorded; camera-only perception results in rain are flagged for manual review.",
    "[TP-07] Software builds used on public roads must pass the full simulation regression suite of 1,200 scenarios with zero critical failures.",
    "[TP-08] Test data is uploaded to the central data lake within 24 hours; raw lidar files are retained for 90 days.",
]
drive_db = FAISS.from_texts(drive_docs, embedding_model)
guard_template = """<|user|>
Relevant information:
{context}

Answer the question using ONLY the information above and cite the rule ID like [TP-01].
If the information does not contain the answer, reply exactly: "Not found in the test procedures."
{question}<|end|>
<|assistant|>"""
guard_prompt = PromptTemplate(template=guard_template, input_variables=["context", "question"])
drive_rag = RunnableParallel(docs=drive_db.as_retriever(search_kwargs={"k": 2}), question=RunnablePassthrough()).assign(
    answer=(lambda x: {"context": join_docs(x["docs"]), "question": x["question"]}) | guard_prompt | llm
)
for q in ["How fast are school-zone scenarios tested?",
          "What must be logged after a disengagement?",
          "What is the maximum allowed speed for snow tests?"]:      # 문서에 없는 질문
    o = drive_rag.invoke(q)
    print(f"Q: {q}\n  retrieved: {[d.page_content[:7] for d in o['docs']]}\n  A: {o['answer'].strip()}\n")

## 5. 고급 RAG 기법과 RAG 평가 (PDF 355–358)

책이 소개하는 개선 기법입니다. 대부분 "검색 전에 LLM을 한 번 더 써서 질문을 다듬는" 형태입니다.

| 기법 | 하는 일 | 예 |
|---|---|---|
| 질의 재작성(query rewriting) | 장황하거나 앞 대화를 참조하는 질문을 검색용 질의로 바꿈 | 긴 사연 → "Where do dolphins live" |
| 다중 질의(multi-query) | 질문 하나를 여러 검색 질의로 나눔 | 2020년 vs 2023년 실적 → 연도별 질의 2개 |
| 다중 홉(multi-hop) | 앞 검색 결과를 보고 다음 질의를 만듦 | 상위 자동차 제조사 찾기 → 각 사의 EV 검색 |
| 질의 라우팅(query routing) | 질문에 따라 검색할 데이터 소스를 고름 | 인사 질문은 HR 문서, 고객 질문은 CRM |
| 에이전틱 RAG(agentic RAG) | 위 판단을 LLM 에이전트가 스스로 함 (7장 ReAct) | 검색을 도구 중 하나로 두고 필요할 때 호출 |

**RAG 평가 축(책 357–358쪽).** 사람 또는 LLM 심판(LLM-as-a-judge)이 평가합니다.

| 축 | 질문 |
|---|---|
| 유창성(fluency) | 답이 자연스럽고 읽기 쉬운가 |
| 체감 유용성(perceived utility) | 질문에 도움이 되는 답인가 |
| 출처 재현율(citation recall) | 사실 주장 중 출처로 뒷받침되는 비율 |
| 출처 정밀도(citation precision) | 단 출처 중 실제로 그 주장을 뒷받침하는 비율 |
| 충실도(faithfulness, Ragas) | 답이 주어진 문맥과 모순 없이 일관된가 |
| 답변 관련성(answer relevance, Ragas) | 답이 질문과 관련 있는가 |

## 6. 정리

| 단계 | 방식 | 강점 | 약점 |
|---|---|---|---|
| 1차 검색 | BM25 (키워드) | 고유명사·정확한 문자열에 강함, 빠름 | 동의어·의역을 못 찾음 |
| 1차 검색 | 밀집 검색 (바이 인코더) | 뜻으로 찾음 | 답이 없어도 무언가 돌려줌, 정확한 문자열에 약함 |
| 2차 정렬 | 리랭커 (크로스 인코더) | 관련도 판단이 정확 | 느려서 후보 수십 개에만 사용, 1차가 놓친 문서는 못 살림 |
| 생성 | RAG | 근거·출처 제시, 최신·사내 지식 반영 | 검색이 틀리면 답도 틀림, 환각이 0은 아님 |

**자율주행·Agent로 이어지는 점(책 밖).** 주행 데이터 플랫폼에서 "비 오는 밤 비보호 좌회전 장면" 같은 씬 검색(scene search)은 이 장의 밀집 검색을 이미지·영상 임베딩으로 넓힌 것입니다(이미지·텍스트 공용 임베딩은 9장 CLIP). 사내 시험 규정·이슈 이력·코드 문서를 근거로 답하는 엔지니어링 비서는 이 장의 RAG이고, 여기에 검색을 도구로 쥐어 준 것이 7장 에이전트와 결합한 에이전틱 RAG입니다. 규모가 커지면 FAISS 대신 관리형 벡터 DB(예: Amazon OpenSearch Service의 벡터 검색)를 씁니다.

**끝나면** Studio에서 space를 **Stop** 하세요. GGUF와 임베딩 모델은 `~/models`와 Hugging Face 캐시에 남아 다음 실행에 재사용됩니다.